6. Detecting Discrepancies
Steps : 
1. Combined four data files to create single data linking each users signup country to the countries of their click.
2. Filtered this combined data to produce a report of users who clicked from a country different from their signup location, flagging them for fraud review.

In [19]:
import pandas as pd
import requests

In [20]:
def load_data(file_path, delimiter=','):
    try:
        return pd.read_csv(file_path, delimiter=delimiter)
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return None

In [21]:
def fetch_country_data(url):
    try:
        response = requests.get(url)
        response.raise_for_status()
        return pd.DataFrame(response.json())
    except requests.exceptions.RequestException as e:
        print(f"Error fetching country data from URL: {e}")
        return None

In [22]:
def find_location_discrepancies():
    
    clicks_df = load_data('7_2_clicks.csv')
    clicks_country_df = load_data('7_1_clicks_country.csv')
    signup_df = load_data('7_4_user_signup_location.csv', delimiter=r'\s*,\s*')
    country_url = 'https://dashboard.pubscale.com/json/country.json'
    country_map_df = fetch_country_data(country_url)

    if clicks_df is None or clicks_country_df is None or signup_df is None or country_map_df is None:
        return

    clicks_country_df['country_code'] = clicks_country_df['country_code'].astype(str)
    signup_df['country_code'] = signup_df['country_code'].astype(str)
    country_map_df['code'] = country_map_df['code'].astype(str)

    user_clicks_df = pd.merge(clicks_df, clicks_country_df, on='click_id')
    signup_df.rename(columns={'country_code': 'signup_country'}, inplace=True)
    user_activity_df = pd.merge(user_clicks_df, signup_df, on='adv_id')
    
    discrepancies_df = user_activity_df[user_activity_df['country_code'] != user_activity_df['signup_country']].copy()

    if not discrepancies_df.empty:
        country_map_df.rename(columns={'code': 'country_code', 'name': 'click_country_name'}, inplace=True)
        discrepancies_df = pd.merge(discrepancies_df, country_map_df[['country_code', 'click_country_name']], on='country_code', how='left')

        country_map_df.rename(columns={'country_code': 'signup_country', 'click_country_name': 'signup_country_name'}, inplace=True)
        discrepancies_df = pd.merge(discrepancies_df, country_map_df[['signup_country', 'signup_country_name']], on='signup_country', how='left')
    
    print("Users with Signup vs. Click Location Discrepancies\n")
    if discrepancies_df.empty:
        print("No location discrepancies found.")
    else:
        final_report = discrepancies_df[[
            'adv_id',
            'signup_country_name',
            'click_country_name'
        ]].drop_duplicates().reset_index(drop=True)

        final_report.fillna('Unknown or Invalid Country', inplace=True)
   

        print(final_report.head(20))

In [23]:
if __name__ == "__main__":
    find_location_discrepancies()

C:\Users\komal\AppData\Local\Temp\ipykernel_20904\715280321.py:3: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  return pd.read_csv(file_path, delimiter=delimiter)


Users with Signup vs. Click Location Discrepancies

                                  adv_id signup_country_name  \
0   5fc3f485-7293-400f-b738-8c74f7df93d2               India   
1   d424a0db-d204-4fc9-aa44-7e99fd5a2993               India   
2   7a0a4061-0b2c-408f-9344-ea7406b678f9               India   
3   3124efe0-390d-4185-ab95-b38ef2eac3bf               India   
4   3c14e8dc-29a5-4e2b-815f-394e1cc444a4           Indonesia   
5   f643b5f7-0f94-4a4e-b99d-eada4ef56632           Indonesia   
6   cadabc3b-7d5d-4411-8a30-4623a177ab3d               India   
7   bbba21c2-0588-47d0-872b-dc9c32598423               India   
8   84610e8a-a91b-44e8-aab5-ca5ccccb6b54               India   
9   d9ff58f9-04c4-4b46-923f-25058eba282a               India   
10  e8abd08e-bdd6-4cb4-8ef9-ccda2ed5f524               India   
11  bd58fc49-9913-4e3f-bdad-411ca697ff8f               India   
12  dea9c325-d07a-4324-8cd1-1c5b568d03c1               India   
13  53bfaaeb-d17d-41e5-8e1f-181144e7e00d            